# HCDE 530 — Week 5: App reviews exploration

This notebook loads **`app_reviews_demo.csv`** and walks through five questions about the data. Run each cell with **Shift + Enter** and read the output before moving on.

---

## Setup

Import pandas and load the CSV. The working directory should be the folder that contains `app_reviews_demo.csv` (for example, open the notebook from **`HCDE 530 Week 5 Project`**).

In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('app_reviews_demo.csv', encoding='utf-8')
print('pandas version:', pd.__version__)
print('rows, columns:', df.shape)

---
## 1. What does your dataset look like? `head()`, `info()`

**`head()`** shows column names and a sample of rows so you can sanity-check values and formatting. **`info()`** lists each column’s dtype, how many non-null values there are, and memory use — your map of what is in the table.

In [ ]:
df.head(10)

In [ ]:
df.info()

---
## 2. What’s the distribution of your most important column?

For app reviews, **`rating`** (1–5) is the main outcome: it tells you how satisfied reviewers are. **`value_counts()`** shows how often each rating appears; sorting by the index puts stars in order.

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()
rating_counts

In [ ]:
rating_pct = df['rating'].value_counts(normalize=True).sort_index().mul(100).round(1)
rating_pct.astype(str) + '%'

---
## 3. Filter to a meaningful subset. What’s in it?

Here we keep only **low ratings (1 or 2)** — the reviews that usually signal problems worth investigating. That is a **meaningful subset** because it separates critical feedback from neutral or positive noise.

We store the result in **`low_ratings`** and inspect shape and a few rows.

In [ ]:
low_ratings = df[df['rating'] <= 2].copy()
print('Subset size:', low_ratings.shape[0], 'rows out of', len(df))
low_ratings.head(10)

In [ ]:
low_ratings[['app', 'category', 'rating', 'review']].head(15)

---
## 4. Group by a category and find the average of a numeric column

We group by **`category`** (the type of product) and take the **mean `rating`** per category. That answers: *which kinds of apps tend to score higher or lower in this sample?*

In [ ]:
df.groupby('category')['rating'].mean().round(2).sort_values(ascending=False)

In [ ]:
df.groupby('category')['rating'].agg(['mean', 'count', 'std']).round(2)

---
## 5. Where are the missing values? Are any columns incomplete?

**`isnull().sum()`** counts missing cells per column. Dividing by row count gives the **share missing**. Any column with a count above zero is **incomplete** for at least some rows — you then decide whether to impute, drop, or analyze missingness separately.

In [ ]:
missing = df.isnull().sum()
missing

In [ ]:
pct_missing = df.isnull().mean().mul(100).round(1)
pct_missing[pct_missing > 0]

In [ ]:
incomplete = missing[missing > 0]
if incomplete.empty:
    print('No missing values in any column.')
else:
    print('Columns with at least one missing value:')
    print(incomplete.to_string())